# Phase I — MFuser-inspired road masks (DINOv2 + CLIP)

Adaptation of [MFuser](https://arxiv.org/abs/2504.03193) (CVPR 2025 Highlight) for DeepGlobe binary roads.

**Paper idea:** fuse a **Vision Foundation Model** (DINOv2) with a **Vision-Language Model** (CLIP) for better domain generalization under canopy / occlusion.

This Colab notebook is a **lightweight MFuser-style** trainer (DINOv2 + CLIP + fusion decoder). Full official code: [devinxzhang/MFuser](https://github.com/devinxzhang/MFuser).

Outputs: `best_road_model_mfuser.pth`, `outputs/masks_mfuser/`


## Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Install packages


In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "accelerate", "albumentations", "opencv-python", "tqdm",
    "segmentation-models-pytorch",
], check=True)
print("ok")


## Paths & split


In [ ]:
import os, random

DRIVE_BASE = "/content/drive/MyDrive/RouteResilience"
SAVE_DIR = f"{DRIVE_BASE}/checkpoints"
DATA_DIR = f"{DRIVE_BASE}/datasets/train"
MASK_DIR = f"{DRIVE_BASE}/outputs/masks_mfuser"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

CKPT_PATH = f"{SAVE_DIR}/best_road_model_mfuser.pth"
MODEL_TAG = "mfuser"

IMG_SIZE = 518   # DINOv2 patch size 14
BATCH_SIZE = 2   # T4: keep 2; try 1 if OOM
EPOCHS = 30
LR_BACKBONE = 1e-5
LR_HEAD = 1e-4
FREEZE_BACKBONE_EPOCHS = 3

TEXT_PROMPTS = ["aerial road", "dirt road", "paved road", "background", "forest", "building"]

all_files = os.listdir(DATA_DIR)
sat = {f.replace("_sat.jpg", "") for f in all_files if f.endswith("_sat.jpg")}
mask = {f.replace("_mask.png", "") for f in all_files if f.endswith("_mask.png")}
all_ids = sorted(sat & mask)
random.seed(42)
random.shuffle(all_ids)
split = int(0.8 * len(all_ids))
train_ids, val_ids = all_ids[:split], all_ids[split:]
print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")
print(f"Checkpoint: {CKPT_PATH}")


## Dataset


In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transforms(img_size=518):
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Resize(img_size, img_size),
        A.RandomShadow(shadow_roi=(0,0,1,1), num_shadows_limit=(1,3),
                       shadow_intensity_range=(0.3,0.7), p=0.5),
        A.CoarseDropout(num_holes_range=(1,8), hole_height_range=(16,48),
                        hole_width_range=(16,48), fill=0, p=0.5),
        A.RandomBrightnessContrast(p=0.4),
        A.HueSaturationValue(p=0.3),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=518):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

class DeepGlobeDataset(Dataset):
    def __init__(self, data_dir, ids, transform=None):
        self.data_dir = data_dir
        self.ids = ids
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = cv2.cvtColor(cv2.imread(f"{self.data_dir}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
        mrgb = cv2.cvtColor(cv2.imread(f"{self.data_dir}/{img_id}_mask.png"), cv2.COLOR_BGR2RGB)
        mask = (mrgb[:, :, 0] > 127).astype(np.float32)
        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]
        return img, mask.unsqueeze(0)

print(f"dataset: {len(train_ids)} train, {len(val_ids)} val")


## Model — MFuser-lite

| Module | Paper | This notebook |
|--------|-------|---------------|
| VFM | DINOv2 | `facebook/dinov2-base` |
| VLM | CLIP | `openai/clip-vit-base-patch32` |
| Fusion | MVFuser (Mamba) | Gated conv fusion (Colab-friendly) |
| Text refine | MTEnhancer | Road/background text bias map |
| Decoder | Seg head | Light FPN head |


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, CLIPModel, CLIPProcessor
import segmentation_models_pytorch as smp

class GatedFusion(nn.Module):
    def __init__(self, c1, c2, out):
        super().__init__()
        self.p1 = nn.Conv2d(c1, out, 1)
        self.p2 = nn.Conv2d(c2, out, 1)
        self.gate = nn.Sequential(nn.Conv2d(out * 2, out, 1), nn.Sigmoid())

    def forward(self, f1, f2):
        f2 = F.interpolate(f2, size=f1.shape[-2:], mode="bilinear", align_corners=False)
        a, b = self.p1(f1), self.p2(f2)
        g = self.gate(torch.cat([a, b], dim=1))
        return a * g + b * (1 - g)


class FPNDecoder(nn.Module):
    def __init__(self, in_dim=256, num_classes=1):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_dim, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, num_classes, 1),
        )

    def forward(self, x, out_size):
        x = F.interpolate(self.head(x), size=out_size, mode="bilinear", align_corners=False)
        return x


class MFuserLite(nn.Module):
    """DINOv2 (VFM) + CLIP (VLM) fusion for binary road masks."""

    def __init__(self, freeze_backbones=True):
        super().__init__()
        self.dino = AutoModel.from_pretrained("facebook/dinov2-base")
        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        self.clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

        self.dino_dim = self.dino.config.hidden_size
                self.clip_dim = self.clip.config.vision_config.hidden_size
        self.patch = self.dino.config.patch_size

        n_layers = self.dino.config.num_hidden_layers
        self.layer_id = n_layers - 1

        self.fuse = GatedFusion(self.dino_dim, self.clip_dim, 256)
        self.decoder = FPNDecoder(256, 1)
        self.patch_proj = nn.Linear(self.clip_dim, self.clip.config.projection_dim)

        # MTEnhancer-lite: precompute text embeddings
        with torch.no_grad():
            tok = self.clip_proc(text=TEXT_PROMPTS, return_tensors="pt", padding=True)
            self.register_buffer(
                "text_emb",
                F.normalize(self.clip.get_text_features(**tok), dim=-1),
            )
        self.road_idx = [0, 1, 2]
        self.bg_idx = [3, 4, 5]

        if freeze_backbones:
            for p in self.dino.parameters():
                p.requires_grad = False
            for p in self.clip.parameters():
                p.requires_grad = False

    def set_backbones_trainable(self, trainable: bool):
        for p in self.dino.parameters():
            p.requires_grad = trainable
        for p in self.clip.parameters():
            p.requires_grad = trainable

    def _dino_map(self, images):
        b, _, h, w = images.shape
        out = self.dino(pixel_values=images, output_hidden_states=True)
        tok = out.hidden_states[self.layer_id + 1][:, 1:, :]
        gh, gw = h // self.patch, w // self.patch
        return tok.reshape(b, gh, gw, -1).permute(0, 3, 1, 2).contiguous()

    def _clip_map(self, images):
        # CLIP vision path at 224, upsample feature map
        x = F.interpolate(images, size=(224, 224), mode="bilinear", align_corners=False)
        vis = self.clip.vision_model(pixel_values=x, output_hidden_states=True)
        tok = vis.last_hidden_state[:, 1:, :]  # patches
        b, n, c = tok.shape
        side = int(n ** 0.5)
                fmap = tok.reshape(b, side, side, c).permute(0, 3, 1, 2)
        return fmap

    def _text_bias(self, clip_map):
        # MTEnhancer-lite: patch-text similarity (road vs background prompts)
        b, c, h, w = clip_map.shape
        f = clip_map.permute(0, 2, 3, 1).reshape(b * h * w, c)
        f = F.normalize(self.patch_proj(f), dim=-1).reshape(b, h, w, -1)
        road = self.text_emb[self.road_idx].mean(0)
        bg = self.text_emb[self.bg_idx].mean(0)
        sim_r = (f * road).sum(-1)
        sim_b = (f * bg).sum(-1)
        return (sim_r - sim_b).unsqueeze(1)

    def forward(self, images):
        h, w = images.shape[-2:]
        dino_f = self._dino_map(images)
        clip_f = self._clip_map(images)
        fused = self.fuse(dino_f, clip_f)
        fused = fused + 0.25 * self._text_bias(clip_f)
        return self.decoder(fused, (h, w))


def build_model(freeze_backbones=True):
    model = MFuserLite(freeze_backbones=freeze_backbones)
    n_all = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"MFuser-lite | total: {n_all:,} | trainable: {n_train:,}")
    return model


class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.4):
        super().__init__()
        self.alpha = alpha
        self.bce = smp.losses.SoftBCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)

    def forward(self, preds, targets):
        return self.alpha * self.bce(preds, targets) + (1 - self.alpha) * self.dice(preds, targets)


def calculate_metrics(preds, targets, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).float()
    targets = targets.float()
    inter = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    iou = (inter + 1e-6) / (union - inter + 1e-6)
    dice = (2.0 * inter + 1e-6) / (union + 1e-6)
    return iou.mean().item(), dice.mean().item()


## Training


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

train_loader = DataLoader(
    DeepGlobeDataset(DATA_DIR, train_ids, get_train_transforms(IMG_SIZE)),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    DeepGlobeDataset(DATA_DIR, val_ids, get_val_transforms(IMG_SIZE)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)

model = build_model(freeze_backbones=True).to(device)
loss_fn = CombinedLoss(0.4)

def make_optimizer(model):
    bb = [p for n, p in model.named_parameters() if p.requires_grad and ("dino" in n or "clip" in n)]
    head = [p for n, p in model.named_parameters() if p.requires_grad and ("dino" not in n and "clip" not in n)]
    groups = [{"params": head, "lr": LR_HEAD}]
    if bb:
        groups.append({"params": bb, "lr": LR_BACKBONE})
    return optim.AdamW(groups, weight_decay=0.05)

optimizer = make_optimizer(model)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

best_val_iou = 0.0
history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
backbone_unfrozen = False

print(f"\nTraining {EPOCHS} epochs on {device}...\n")
for epoch in range(1, EPOCHS + 1):
    if (not backbone_unfrozen) and epoch > FREEZE_BACKBONE_EPOCHS:
        model.set_backbones_trainable(True)
        optimizer = make_optimizer(model)
        backbone_unfrozen = True
        print(f"Epoch {epoch}: unfroze DINOv2 + CLIP")

    model.train()
    t_loss = t_iou = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        images = images.to(device, torch.float32)
        masks = masks.to(device, torch.float32)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
        iou, _ = calculate_metrics(outputs, masks)
        t_iou += iou

    model.eval()
    v_loss = v_iou = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]"):
            images = images.to(device, torch.float32)
            masks = masks.to(device, torch.float32)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            v_loss += loss.item()
            iou, _ = calculate_metrics(outputs, masks)
            v_iou += iou

    t_loss /= len(train_loader); t_iou /= len(train_loader)
    v_loss /= len(val_loader); v_iou /= len(val_loader)
    scheduler.step(v_loss)
    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_iou"].append(t_iou)
    history["val_iou"].append(v_iou)

    print(f"Epoch {epoch:02d} | Train Loss: {t_loss:.4f} IoU: {t_iou:.4f} | Val Loss: {v_loss:.4f} IoU: {v_iou:.4f}")
    if v_iou > best_val_iou:
        best_val_iou = v_iou
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  new best val iou {v_iou:.4f}")

print(f"\nfinished. best val iou {best_val_iou:.4f}")
print(f"Checkpoint: {CKPT_PATH}")


## Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train", color="tomato")
axes[0].plot(history["val_loss"], label="Val", color="steelblue")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history["train_iou"], label="Train", color="tomato")
axes[1].plot(history["val_iou"], label="Val", color="steelblue")
axes[1].set_title("IoU"); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves_{MODEL_TAG}.png", dpi=120)
plt.show()
print(f"Best Val IoU: {best_val_iou:.4f}")


## Inference


In [ ]:
model = build_model(freeze_backbones=False).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()
print("Loaded", CKPT_PATH)

@torch.no_grad()
def predict_and_save(img_id):
    img = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    aug = get_val_transforms(IMG_SIZE)(image=img, mask=np.zeros(img.shape[:2], dtype=np.float32))
    t = aug["image"].unsqueeze(0).to(device)
    logits = model(t)
    pred = (torch.sigmoid(logits) > 0.5).squeeze().cpu().numpy().astype(np.uint8) * 255
    if pred.shape != (h, w):
        pred = cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(f"{MASK_DIR}/{img_id}_pred.png", pred)
    return pred

DEMO_IDS = ["493626", "477671", "422265", "194764"]
demo_ids = [i for i in DEMO_IDS if i in val_ids or i in train_ids] or val_ids[:4]
for img_id in demo_ids:
    predict_and_save(img_id)
    print(f"Saved {MASK_DIR}/{img_id}_pred.png")
print("Done.")


## Quick visual check


In [ ]:
sample_ids = demo_ids[:4]
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(12, 4 * len(sample_ids)))
if len(sample_ids) == 1:
    axes = axes.reshape(1, -1)
for row, img_id in enumerate(sample_ids):
    sat = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
    gt = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_mask.png"), cv2.COLOR_BGR2RGB)
    pred = cv2.imread(f"{MASK_DIR}/{img_id}_pred.png", cv2.IMREAD_GRAYSCALE)
    axes[row,0].imshow(sat); axes[row,0].set_title(img_id); axes[row,0].axis("off")
    axes[row,1].imshow(gt); axes[row,1].set_title("GT"); axes[row,1].axis("off")
    axes[row,2].imshow(pred, cmap="gray"); axes[row,2].set_title("MFuser-lite"); axes[row,2].axis("off")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/predictions_{MODEL_TAG}.png", dpi=120)
plt.show()
